Create your catalog and schemas (Unity Catalog):

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS elevator")
spark.sql("CREATE SCHEMA IF NOT EXISTS elevator.raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS elevator.clean")
spark.sql("CREATE SCHEMA IF NOT EXISTS elevator.model")

DataFrame[]

In [0]:
raw_comp = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/contract_review/compliance/raw/DOB_NOW_Elevator_Safety_Compliance_20260927.csv"))

raw_viol = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/contract_review/compliance/raw/DOB_Safety_Violations_20260927.csv"))

In [0]:
raw_comp.printSchema()
print("Compliance rows:", raw_comp.count())
display(raw_comp.limit(10))

root
 |-- Device Number: string (nullable = true)
 |-- Device Type: string (nullable = true)
 |-- Device Status: string (nullable = true)
 |-- Status Date: date (nullable = true)
 |-- Equipment Type: string (nullable = true)
 |-- Periodic Report Year: integer (nullable = true)
 |-- CAT1 Report Year: integer (nullable = true)
 |-- CAT1 Latest Report Filed Date: date (nullable = true)
 |-- CAT5 Latest Report Filed Date: date (nullable = true)
 |-- Periodic Latest Inspection Date: date (nullable = true)
 |-- BIN: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- House Number: string (nullable = true)
 |-- Street Name: string (nullable = true)
 |-- Block: integer (nullable = true)
 |-- Lot: integer (nullable = true)
 |-- Postcode: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Community Board: integer (nullable = true)
 |-- Council District: integer (nullable = true)
 |-- BBL: long (nullable = true)
 |-- Ce

Device Number,Device Type,Device Status,Status Date,Equipment Type,Periodic Report Year,CAT1 Report Year,CAT1 Latest Report Filed Date,CAT5 Latest Report Filed Date,Periodic Latest Inspection Date,BIN,Borough,House Number,Street Name,Block,Lot,Postcode,Latitude,Longitude,Community Board,Council District,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
1P14518,Elevator,Active,2021-12-29,null,2025,2025,2025-09-25,2023-04-19,2025-06-26,1001627,Manhattan,85,CHAMBERS STREET,149,6,10007.0,40.714535,-74.007081,101,1,1001490006,21,Tribeca-Civic Center
4P1000100,Elevator,Work in Progress,2024-07-25,null,null,null,null,null,null,4143518,Queens,141-32,73 AVENUE,6624,17,11367.0,40.724301,-73.821978,408,24,4066240017,77902,Kew Gardens Hills
2W12120,Accessibility Lift,Removed,2010-03-23,null,null,2014,2014-06-09,2010-06-14,null,2017315,Bronx,3005,GRAND CONCOURSE,3310,72,10468.0,40.873151,-73.887197,207,14,2033100072,411,Bedford Park
3F2219,Elevator,Removed,2013-11-27,null,null,null,null,null,null,3329396,Brooklyn,139,PLYMOUTH STREET,18,1,11201.0,40.703811,-73.987925,302,33,3000180001,21,Downtown Brooklyn-DUMBO-Boerum Hill
1P43871,Elevator,Removed,2024-09-11,null,2022,2022,2022-12-29,2017-10-12,2022-05-18,1086548,Manhattan,123,BAXTER STREET,206,7501,10013.0,40.718161,-73.998918,102,1,1002067501,41,SoHo-Little Italy-Hudson Square
1P8115,Elevator,Active,2020-05-04,null,2024,2024,2024-12-06,2020-05-08,2024-08-09,1083724,Manhattan,147,WEST 55 STREET,1008,1,10019.0,40.763898,-73.980189,105,6,1010080001,137,Midtown-Times Square
1P16074,Elevator,Active,2021-10-02,null,2026,2026,2026-04-18,2025-04-24,2026-08-10,1015254,Manhattan,1375,BROADWAY,813,16,10018.0,40.752328,-73.987461,105,3,1008130016,109,Midtown-Times Square
1W548,Accessibility Lift,Active,1995-11-14,null,2025,2026,2026-09-17,null,2025-07-08,1031607,Manhattan,300,CENTRAL PARK WEST,1204,29,10024.0,40.788063,-73.967315,107,6,1012040029,177,Upper West Side (Central)
1P23480,Elevator,Active,1991-11-01,null,2026,2025,2025-09-20,2023-04-18,2026-06-03,1036206,Manhattan,485,LEXINGTON AVENUE,1301,23,10017.0,40.754085,-73.974114,106,4,1013010023,92,East Midtown-Turtle Bay
1P0986603,Elevator,Active,2018-09-21,ElectricElevators,2026,2025,2025-06-12,2024-11-14,2026-05-27,1024727,Manhattan,1535,BROADWAY,1017,29,10036.0,40.758267,-73.985468,105,3,1010177501,119,Midtown-Times Square


In [0]:
raw_viol.printSchema()
print("Violation rows:", raw_viol.count())
display(raw_viol.limit(10))

root
 |-- BIN: integer (nullable = true)
 |-- Violation Issue Date: date (nullable = true)
 |-- Violation Number: string (nullable = true)
 |-- Violation Type: string (nullable = true)
 |-- Violation Remarks: string (nullable = true)
 |-- Violation Status: string (nullable = true)
 |-- Device Number: string (nullable = true)
 |-- Device Type: string (nullable = true)
 |-- Cycle End Date: date (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Block: integer (nullable = true)
 |-- Lot: string (nullable = true)
 |-- House Number: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postcode: integer (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Community Board: integer (nullable = true)
 |-- Council District: integer (nullable = true)
 |-- BBL: long (nullable = true)
 |-- Census Tract (2020): integer (nullable = true)
 |-- Neighborho

BIN,Violation Issue Date,Violation Number,Violation Type,Violation Remarks,Violation Status,Device Number,Device Type,Cycle End Date,Borough,Block,Lot,House Number,Street,City,State,Postcode,Latitude,Longitude,Community Board,Council District,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
1055894,2026-09-25,VIO-DQ-EN-BENCH-202608-0166309,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1851,16,119,WEST 96 STREET,New York,New York,10025,40.793198,-73.968313,107,7,1018510016,181,Upper West Side (Central)
1081126,2026-09-25,VIO-FTC-AEU-HAZ-202609-0040921,FTC-AEU-HAZ,null,Active,null,AEUHAZ,null,Manhattan,1262,9,59,WEST 46 STREET,New York,New York,10036,40.756803,-73.980841,105,4,1012620009,96,Midtown-Times Square
5091139,2026-09-25,VIO-DQ-EN-BENCH-202608-0167587,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Staten Island,3223,12,111,QUINTARD STREET,New York,New York,10305,40.594493,-74.08317,502,50,5032230012,64,Grasmere-Arrochar-South Beach-Dongan Hills
1076524,2026-09-25,VIO-DQ-EN-BENCH-202608-0166306,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1846,"7,501",2031,FRED DOUGLASS BOULEVARD,New York,New York,10026,40.801086,-73.957816,110,9,1018467501,19702,Harlem (South)
1085962,2026-09-25,VIO-DQ-EN-BENCH-202608-0166123,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1089,"7,502",600,WEST 42 STREET,New York,New York,10036,40.760884,-73.998477,104,3,1010897502,117,Chelsea-Hudson Yards
4308806,2026-09-25,VIO-DQ-EN-BENCH-202608-0167407,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Queens,4977,"7,502",136-02,NORTHERN BOULEVARD,New York,New York,11354,40.762976,-73.831838,407,20,4049777502,869,Flushing-Willets Point
2015416,2026-09-25,VIO-DQ-EN-BENCH-202608-0166721,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Bronx,3250,65,2844,WEBB AVENUE,New York,New York,10468,40.873141,-73.900687,208,14,2032500065,26702,Kingsbridge Heights-Van Cortlandt Village
3397090,2026-09-25,VIO-FTF-EN-BENCH-202608-0166696,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Brooklyn,6910,10,8895,26 AVENUE,New York,New York,11214,40.59071,-73.991164,313,47,3069107501,31401,Gravesend (South)
1017107,2026-09-25,VIO-FTF-EN-BENCH-202608-0166328,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,864,47,34,EAST 35 STREET,New York,New York,10016,40.748248,-73.982056,106,4,1008640047,74,Midtown South-Flatiron-Union Square
3053589,2026-09-25,VIO-FTF-EN-BENCH-202608-0166603,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Brooklyn,1856,41,444,MARCUS GARVEY AVENUE,New York,New York,11216,40.681804,-73.93782,303,36,3018560041,273,Bedford-Stuyvesant (East)


Writing the bronze tables

In [0]:
from pyspark.sql import functions as F

raw_comp = (spark.read.option("header", True).option("inferSchema", True)
    .csv("/Volumes/contract_review/compliance/raw/DOB_NOW_Elevator_Safety_Compliance_20260927.csv"))

# rename spaced columns
raw_comp = (raw_comp
    .withColumnRenamed("Device Number","Device_Number")
    .withColumnRenamed("Device Type","Device_Type")
    .withColumnRenamed("Device Status","Device_Status")
    .withColumnRenamed("CAT1 Latest Report Filed Date","CAT1_Filed_Date")
    .withColumnRenamed("CAT5 Latest Report Filed Date","CAT5_Filed_Date")
    .withColumnRenamed("Periodic Latest Inspection Date","Periodic_Inspection_Date")
    .withColumnRenamed("CAT1 Report Year","CAT1_Report_Year")
    .withColumnRenamed("House Number","House_Number")
    .withColumnRenamed("Street Name","Street_Name")
    .withColumnRenamed("Status Date","Status_Date")
    .withColumnRenamed("Equipment Type","Equipment_Type")
    .withColumnRenamed("Periodic Report Year","Periodic_Report_Year")
    .withColumnRenamed("Community Board","Community_Board")
    .withColumnRenamed("Council District","Council_District")
    .withColumnRenamed("Census Tract (2020)","Census_Tract_2020")
    .withColumnRenamed("Neighborhood Tabulation Area (NTA) (2020)","Neighborhood_Tabulation_Area_NTA_2020"))

raw_comp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("elevator.raw.compliance")

raw_viol = (spark.read.option("header", True).option("inferSchema", True)
    .csv("/Volumes/contract_review/compliance/raw/DOB_Safety_Violations_20260927.csv"))
raw_viol = (raw_viol
    .withColumnRenamed("Device Number","Device_Number")
    .withColumnRenamed("Violation Type","Violation_Type")
    .withColumnRenamed("Violation Status","Violation_Status")
    .withColumnRenamed("Violation Issue Date","Violation_Issue_Date")
    .withColumnRenamed("Device Type","Device_Type")
    .withColumnRenamed("Violation Number","Violation_Number")
    .withColumnRenamed("Violation Remarks","Violation_Remarks")
    .withColumnRenamed("Cycle End Date","Cycle_End_Date")
    .withColumnRenamed("House Number","House_Number")
    .withColumnRenamed("Community Board","Community_Board")
    .withColumnRenamed("Council District","Council_District")
    .withColumnRenamed("Census Tract (2020)","Census_Tract_2020")
    .withColumnRenamed("Neighborhood Tabulation Area (NTA) (2020)","Neighborhood_Tabulation_Area_NTA_2020"))
raw_viol.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("elevator.raw.violations")

Cleaning the tables (Silver Layer)

In [0]:
from pyspark.sql import functions as F
clean_comp = (spark.table("elevator.raw.compliance")
    .withColumn("Device_Number", F.upper(F.trim("Device_Number")))
    .withColumn("BIN", F.trim(F.col("BIN").cast("string")))
    .withColumn("Device_Status", F.initcap(F.trim("Device_Status")))
    # 1. convert text to real dates
    .withColumn("CAT1_Filed_Date", F.to_date("CAT1_Filed_Date", "MM/dd/yyyy"))
    .withColumn("CAT5_Filed_Date", F.to_date("CAT5_Filed_Date", "MM/dd/yyyy"))
    .withColumn("Periodic_Inspection_Date", F.to_date("Periodic_Inspection_Date", "MM/dd/yyyy"))
    # 2. then null out old/placeholder dates
    .withColumn("CAT1_Filed_Date", F.when(F.col("CAT1_Filed_Date") < F.lit("2020-01-01").cast("date"), None).otherwise(F.col("CAT1_Filed_Date")))
    .withColumn("CAT5_Filed_Date", F.when(F.col("CAT5_Filed_Date") < F.lit("2000-01-01").cast("date"), None).otherwise(F.col("CAT5_Filed_Date")))
    .filter(F.col("Device_Number").isNotNull())
    .dropDuplicates(["Device_Number"]))

clean_comp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("elevator.clean.compliance")

In [0]:
display(spark.table("elevator.raw.compliance").limit(10))

Device_Number,Device_Type,Device_Status,Status_Date,Equipment_Type,Periodic_Report_Year,CAT1_Report_Year,CAT1_Filed_Date,CAT5_Filed_Date,Periodic_Inspection_Date,BIN,Borough,House_Number,Street_Name,Block,Lot,Postcode,Latitude,Longitude,Community_Board,Council_District,BBL,Census_Tract_2020,Neighborhood_Tabulation_Area_NTA_2020
1P14518,Elevator,Active,2021-12-29,null,2025,2025,2025-09-25,2023-04-19,2025-06-26,1001627,Manhattan,85,CHAMBERS STREET,149,6,10007.0,40.714535,-74.007081,101,1,1001490006,21,Tribeca-Civic Center
4P1000100,Elevator,Work in Progress,2024-07-25,null,null,null,null,null,null,4143518,Queens,141-32,73 AVENUE,6624,17,11367.0,40.724301,-73.821978,408,24,4066240017,77902,Kew Gardens Hills
2W12120,Accessibility Lift,Removed,2010-03-23,null,null,2014,2014-06-09,2010-06-14,null,2017315,Bronx,3005,GRAND CONCOURSE,3310,72,10468.0,40.873151,-73.887197,207,14,2033100072,411,Bedford Park
3F2219,Elevator,Removed,2013-11-27,null,null,null,null,null,null,3329396,Brooklyn,139,PLYMOUTH STREET,18,1,11201.0,40.703811,-73.987925,302,33,3000180001,21,Downtown Brooklyn-DUMBO-Boerum Hill
1P43871,Elevator,Removed,2024-09-11,null,2022,2022,2022-12-29,2017-10-12,2022-05-18,1086548,Manhattan,123,BAXTER STREET,206,7501,10013.0,40.718161,-73.998918,102,1,1002067501,41,SoHo-Little Italy-Hudson Square
1P8115,Elevator,Active,2020-05-04,null,2024,2024,2024-12-06,2020-05-08,2024-08-09,1083724,Manhattan,147,WEST 55 STREET,1008,1,10019.0,40.763898,-73.980189,105,6,1010080001,137,Midtown-Times Square
1P16074,Elevator,Active,2021-10-02,null,2026,2026,2026-04-18,2025-04-24,2026-08-10,1015254,Manhattan,1375,BROADWAY,813,16,10018.0,40.752328,-73.987461,105,3,1008130016,109,Midtown-Times Square
1W548,Accessibility Lift,Active,1995-11-14,null,2025,2026,2026-09-17,null,2025-07-08,1031607,Manhattan,300,CENTRAL PARK WEST,1204,29,10024.0,40.788063,-73.967315,107,6,1012040029,177,Upper West Side (Central)
1P23480,Elevator,Active,1991-11-01,null,2026,2025,2025-09-20,2023-04-18,2026-06-03,1036206,Manhattan,485,LEXINGTON AVENUE,1301,23,10017.0,40.754085,-73.974114,106,4,1013010023,92,East Midtown-Turtle Bay
1P0986603,Elevator,Active,2018-09-21,ElectricElevators,2026,2025,2025-06-12,2024-11-14,2026-05-27,1024727,Manhattan,1535,BROADWAY,1017,29,10036.0,40.758267,-73.985468,105,3,1010177501,119,Midtown-Times Square


In [0]:
# violations: keep only elevator-related, drop energy/benchmarking noise
clean_viol = (spark.table("elevator.raw.violations")
    .withColumn("BIN", F.trim(F.col("BIN").cast("string")))
    .withColumn("Violation_Status", F.initcap(F.trim("Violation_Status")))
    .withColumn("Violation_Issue_Date", F.to_date("Violation_Issue_Date","MM/dd/yyyy"))
    .filter(F.upper(F.col("Device_Type")).contains("ELEV") | F.upper(F.col("Violation_Type")).startswith("EV"))
    .dropDuplicates())
clean_viol.write.format("delta").mode("overwrite").saveAsTable("elevator.clean.violations")

In [0]:
display(spark.table("elevator.raw.violations").limit(10))

BIN,Violation_Issue_Date,Violation_Number,Violation_Type,Violation_Remarks,Violation_Status,Device_Number,Device_Type,Cycle_End_Date,Borough,Block,Lot,House_Number,Street,City,State,Postcode,Latitude,Longitude,Community_Board,Council_District,BBL,Census_Tract_2020,Neighborhood_Tabulation_Area_NTA_2020
1055894,2026-09-25,VIO-DQ-EN-BENCH-202608-0166309,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1851,16,119,WEST 96 STREET,New York,New York,10025,40.793198,-73.968313,107,7,1018510016,181,Upper West Side (Central)
1081126,2026-09-25,VIO-FTC-AEU-HAZ-202609-0040921,FTC-AEU-HAZ,null,Active,null,AEUHAZ,null,Manhattan,1262,9,59,WEST 46 STREET,New York,New York,10036,40.756803,-73.980841,105,4,1012620009,96,Midtown-Times Square
5091139,2026-09-25,VIO-DQ-EN-BENCH-202608-0167587,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Staten Island,3223,12,111,QUINTARD STREET,New York,New York,10305,40.594493,-74.08317,502,50,5032230012,64,Grasmere-Arrochar-South Beach-Dongan Hills
1076524,2026-09-25,VIO-DQ-EN-BENCH-202608-0166306,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1846,"7,501",2031,FRED DOUGLASS BOULEVARD,New York,New York,10026,40.801086,-73.957816,110,9,1018467501,19702,Harlem (South)
1085962,2026-09-25,VIO-DQ-EN-BENCH-202608-0166123,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,1089,"7,502",600,WEST 42 STREET,New York,New York,10036,40.760884,-73.998477,104,3,1010897502,117,Chelsea-Hudson Yards
4308806,2026-09-25,VIO-DQ-EN-BENCH-202608-0167407,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Queens,4977,"7,502",136-02,NORTHERN BOULEVARD,New York,New York,11354,40.762976,-73.831838,407,20,4049777502,869,Flushing-Willets Point
2015416,2026-09-25,VIO-DQ-EN-BENCH-202608-0166721,DQ-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Bronx,3250,65,2844,WEBB AVENUE,New York,New York,10468,40.873141,-73.900687,208,14,2032500065,26702,Kingsbridge Heights-Van Cortlandt Village
3397090,2026-09-25,VIO-FTF-EN-BENCH-202608-0166696,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Brooklyn,6910,10,8895,26 AVENUE,New York,New York,11214,40.59071,-73.991164,313,47,3069107501,31401,Gravesend (South)
1017107,2026-09-25,VIO-FTF-EN-BENCH-202608-0166328,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Manhattan,864,47,34,EAST 35 STREET,New York,New York,10016,40.748248,-73.982056,106,4,1008640047,74,Midtown South-Flatiron-Union Square
3053589,2026-09-25,VIO-FTF-EN-BENCH-202608-0166603,FTF-EN-BENCH,FAILURE TO FILE BENCHMARKING REPORT OF ENERGY USE AS PER AD. CODE SEC. 28-309.4,Active,null,Benchmarking - LL84,null,Brooklyn,1856,41,444,MARCUS GARVEY AVENUE,New York,New York,11216,40.681804,-73.93782,303,36,3018560041,273,Bedford-Stuyvesant (East)


Gold Layer - Fact tables, Dimensions tables

In [0]:
comp = spark.table("elevator.clean.compliance").filter(F.col("Device_Status")=="Active")

# open violations rolled up to BIN (since Device Number is blank in violations)
viol_by_bin = (spark.table("elevator.clean.violations")
    .filter(F.col("Violation_Status")=="Active")
    .groupBy("BIN").agg(F.count("*").alias("Open_Violation_Count")))

fact = (comp
    .join(viol_by_bin, "BIN", "left")
    .withColumn("CAT1_Due_Date", F.expr("make_date(year(CAT1_Filed_Date)+1, 12, 31)"))
    .withColumn("CAT5_Due_Date", F.expr("add_months(CAT5_Filed_Date, 60)"))
    .withColumn("Next_Due_Date", F.least("CAT1_Due_Date","CAT5_Due_Date"))
    .withColumn("Days_Until_Due", F.datediff("Next_Due_Date", F.current_date()))
    .withColumn("Compliance_Status",
        F.when(F.col("Next_Due_Date").isNull(), "No Record")
         .when(F.col("Days_Until_Due") < 0, "Overdue")
         .when(F.col("Days_Until_Due") <= 30, "Due Soon")
         .otherwise("Compliant"))
    .fillna({"Open_Violation_Count": 0})
    .select("Device_Number","BIN","Borough","Device_Type",
            "CAT1_Filed_Date","CAT5_Filed_Date","CAT1_Due_Date","CAT5_Due_Date",
            "Next_Due_Date","Days_Until_Due","Compliance_Status","Open_Violation_Count"))
fact.write.format("delta").mode("overwrite").saveAsTable("elevator.model.fact_compliance")

In [0]:
# Building dimension
dim_building = (spark.table("elevator.clean.compliance")
    .select("BIN","Borough","Street_Name","Postcode","Latitude","Longitude").dropDuplicates(["BIN"]))
dim_building.write.format("delta").mode("overwrite").saveAsTable("elevator.model.dim_building")

# Device dimension
dim_device = (spark.table("elevator.clean.compliance")
    .select("Device_Number","Device_Status").dropDuplicates(["Device_Number"]))
dim_device.write.format("delta").mode("overwrite").saveAsTable("elevator.model.dim_device")

# Date dimension
dim_date = (spark.sql("""
  SELECT explode(sequence(to_date('2015-01-01'), to_date('2030-12-31'), interval 1 day)) AS Date
""")
  .withColumn("Year", F.year("Date"))
  .withColumn("Month", F.month("Date"))
  .withColumn("MonthName", F.date_format("Date","MMMM")))
dim_date.write.format("delta").mode("overwrite").saveAsTable("elevator.model.dim_date")

In [0]:
from pyspark.sql import functions as F

dim_building = (spark.table("elevator.clean.compliance")
    .select("BIN","Borough","House_Number","Street_Name","Postcode","Latitude","Longitude")
    .dropDuplicates(["BIN"]))          # one row per BIN

# confirm it's actually unique
print("rows:", dim_building.count(), "distinct BIN:", dim_building.select("BIN").distinct().count())

dim_building.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("elevator.model.dim_building")

rows: 47155 distinct BIN: 47155


In [0]:
%sql
CREATE OR REPLACE VIEW elevator.model.vw_compliance_queue AS
SELECT
  f.Device_Number,
  f.BIN,
  f.Borough,
  f.Compliance_Status,
  CAST(f.Next_Due_Date AS STRING) AS Next_Due_Date,
  f.Days_Until_Due,
  f.Open_Violation_Count
FROM elevator.model.fact_compliance f
WHERE f.Compliance_Status IN ('Overdue','Due Soon')
ORDER BY f.Days_Until_Due ASC
LIMIT 200;